# 04 — Generative Adversarial Network (GAN) on MNIST Digit Generation
### Deep Learning Week 3: Adversarial Training, Generator-Discriminator Dynamics, and Image Synthesis

---

## 1. Conceptual Introduction: The Minimax Game of GANs

Introduced by Ian Goodfellow et al. (2014), **Generative Adversarial Networks (GANs)** frame generative modeling as a two-player zero-sum game between two competing neural networks:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1 - D(G(z)))]$$

### 1. The Generator ($G$): The Counterfeiter
- Takes a random latent noise vector $z \sim \mathcal{N}(0, I)$ of dimension $d_z = 100$.
- Maps $z$ through convolutional and transpose-convolutional layers to produce a synthetic $28 \times 28$ image: $x_{\text{fake}} = G(z)$.
- Objective: Maximize $D(G(z))$, fooling the Discriminator into classifying $x_{\text{fake}}$ as a genuine handwritten digit. In practice, we minimize $-\log D(G(z))$ to prevent vanishing gradients early in training.

### 2. The Discriminator ($D$): The Detective
- Takes a $28 \times 28$ grayscale image (either real $x$ from the dataset or fake $G(z)$).
- Outputs a scalar score $D(x) \in [0, 1]$ indicating the probability that $x$ came from the true data distribution.
- Objective: Maximize $\log D(x) + \log(1 - D(G(z)))$.

### Critical GAN Challenges
- **Oscillating Loss Curves:** Unlike standard supervised training where loss monotonically decreases, GAN losses fluctuate as the two adversaries adapt to each other's strategies. A low Generator loss is meaningless if the Discriminator is simply collapsing.
- **Mode Collapse:** The Generator discovers a single digit class (e.g., drawing only '1's or '8's) that reliably fools the Discriminator, failing to cover the full diversity of the data distribution.
- **Vanishing Gradients:** If the Discriminator learns too fast, $D(G(z)) \approx 0$ and the Generator receives negligible gradient signal to improve. We use **one-sided label smoothing** ($0.9$ instead of $1.0$ for real labels) and LeakyReLU activations to stabilize training.


In [ ]:
# 2. Imports and Environment Setup
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available:      {len(tf.config.list_physical_devices('GPU')) > 0}")


## 3. Dataset Preprocessing: Normalizing to [-1.0, 1.0]

The Generator uses the `tanh` activation function in its final layer, which bounds pixel outputs to $[-1.0, 1.0]$.
We normalize the MNIST training images to this exact range:
$$x_{\text{norm}} = \frac{x - 127.5}{127.5} \in [-1.0, 1.0]$$


In [ ]:
# 3.1 Load and Normalize MNIST Dataset
(x_train_raw, _), (x_test_raw, _) = keras.datasets.mnist.load_data()

# Normalize to [-1, 1]
x_train = (x_train_raw.astype("float32") - 127.5) / 127.5
x_train = np.expand_dims(x_train, axis=-1)  # Shape: (60000, 28, 28, 1)

print(f"MNIST training set: {x_train.shape}, range: [{x_train.min():.1f}, {x_train.max():.1f}]")


## 4. Architecture Definition: Generator & Discriminator

### Generator Architecture:
- Input: Latent vector $z \in \mathbb{R}^{100}$
- `Dense(7 * 7 * 128)` + `LeakyReLU(0.2)`
- `Reshape((7, 7, 128))`
- `Conv2DTranspose(64, kernel_size=4, strides=2, padding='same')` $\to (14, 14, 64)$ + `LeakyReLU(0.2)`
- `Conv2DTranspose(1, kernel_size=4, strides=2, padding='same', activation='tanh')` $\to (28, 28, 1)$

### Discriminator Architecture:
- Input: Grayscale image $(28, 28, 1)$
- `Conv2D(64, kernel_size=3, strides=2, padding='same')` $\to (14, 14, 64)$ + `LeakyReLU(0.2)` + `Dropout(0.3)`
- `Conv2D(128, kernel_size=3, strides=2, padding='same')` $\to (7, 7, 128)$ + `LeakyReLU(0.2)` + `Dropout(0.3)`
- `Flatten()` $\to 6272$
- `Dense(1, activation='sigmoid')`


In [ ]:
# 4.1 Define the Generator Network
LATENT_DIM = 100

def make_generator():
    model = keras.Sequential([
        layers.Input(shape=(LATENT_DIM,)),
        layers.Dense(7 * 7 * 128),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same"),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Conv2DTranspose(1, kernel_size=4, strides=2, padding="same", activation="tanh"),
    ], name="DCGAN_Generator")
    return model

generator = make_generator()
generator.summary()

# 4.2 Define the Discriminator Network
def make_discriminator():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(64, kernel_size=3, strides=2, padding="same"),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),
        layers.Conv2D(128, kernel_size=3, strides=2, padding="same"),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1, activation="sigmoid"),
    ], name="DCGAN_Discriminator")
    return model

discriminator = make_discriminator()
discriminator.summary()


## 5. Adversarial Training Loop

We configure separate Adam optimizers ($lr=0.0002, \beta_1=0.5$) and execute the two-phase training step:
1. **Discriminator Step:** Train on real samples (label = 0.9 with label smoothing) and fake generated samples (label = 0.0).
2. **Generator Step:** Sample new latent noise vectors and optimize Generator parameters with target label = 1.0.


In [ ]:
# 5.1 Build Adversarial Training Setup
d_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
g_optimizer = keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
bce_loss = keras.losses.BinaryCrossentropy()

# Fixed noise vector to track visual progression across training epochs
FIXED_NOISE = tf.random.normal([16, LATENT_DIM], seed=42)

@tf.function
def train_step(real_images, batch_size):
    # 1. Sample latent noise
    noise = tf.random.normal([batch_size, LATENT_DIM])

    # 2. Train Discriminator
    with tf.GradientTape() as d_tape:
        fake_images = generator(noise, training=True)
        real_output = discriminator(real_images, training=True)
        fake_output = discriminator(fake_images, training=True)

        # One-sided label smoothing on real images (0.9)
        d_loss_real = bce_loss(tf.ones_like(real_output) * 0.9, real_output)
        d_loss_fake = bce_loss(tf.zeros_like(fake_output), fake_output)
        d_loss = d_loss_real + d_loss_fake

    d_gradients = d_tape.gradient(d_loss, discriminator.trainable_variables)
    d_optimizer.apply_gradients(zip(d_gradients, discriminator.trainable_variables))

    # 3. Train Generator (fool discriminator)
    noise_g = tf.random.normal([batch_size, LATENT_DIM])
    with tf.GradientTape() as g_tape:
        generated_images = generator(noise_g, training=True)
        predictions = discriminator(generated_images, training=True)
        g_loss = bce_loss(tf.ones_like(predictions), predictions)

    g_gradients = g_tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(g_gradients, generator.trainable_variables))

    return d_loss, g_loss


## 6. Execution: Adversarial Training with Progress Grids

We train for 12 epochs over a representative subset of the dataset for fast local convergence.


In [ ]:
# 6.1 Train the GAN
BATCH_SIZE = 128
EPOCHS = 12
STEPS_PER_EPOCH = 150  # 150 batches * 128 = 19,200 samples per epoch

dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(5000).batch(BATCH_SIZE, drop_remainder=True)

d_losses = []
g_losses = []
stage_samples = {}

print("Beginning Adversarial Training Loop...")
for epoch in range(1, EPOCHS + 1):
    epoch_d_loss = []
    epoch_g_loss = []
    
    for step, real_batch in enumerate(dataset.take(STEPS_PER_EPOCH)):
        d_l, g_l = train_step(real_batch, BATCH_SIZE)
        epoch_d_loss.append(float(d_l))
        epoch_g_loss.append(float(g_l))

    mean_d = np.mean(epoch_d_loss)
    mean_g = np.mean(epoch_g_loss)
    d_losses.append(mean_d)
    g_losses.append(mean_g)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Discriminator Loss: {mean_d:.4f} | Generator Loss: {mean_g:.4f}")

    # Capture visual snapshots at key milestones
    if epoch in [1, 4, 8, 12]:
        preds = generator(FIXED_NOISE, training=False).numpy()
        # Rescale from [-1, 1] to [0, 1]
        stage_samples[epoch] = (preds + 1.0) / 2.0


## 7. Visual Progression: From Random Noise to Coherent Digits


In [ ]:
# 7.1 Visualize Generated Digits Across Training Epochs
fig, axes = plt.subplots(len(stage_samples), 8, figsize=(14, 2.5 * len(stage_samples)))

for row_idx, (epoch, images) in enumerate(stage_samples.items()):
    for col_idx in range(8):
        ax = axes[row_idx, col_idx]
        ax.imshow(images[col_idx, :, :, 0], cmap="gray")
        if col_idx == 0:
            ax.set_ylabel(f"Epoch {epoch}", fontsize=11, fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("GAN Synthesis Progression from Fixed Latent Vectors", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 8. Loss Curves and Convergence Analysis


In [ ]:
# 8.1 Plot Adversarial Loss History
plt.figure(figsize=(10, 4.5))
plt.plot(d_losses, label="Discriminator Loss", color="#d62728", marker="o")
plt.plot(g_losses, label="Generator Loss", color="#1f77b4", marker="s")
plt.title("Adversarial Loss History Across Training Epochs", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


## 9. Discussion & Critical Analysis

### 1. Visual Progression Analysis:
- **Epoch 1:** Output consists of amorphous Gaussian noise and fuzzy white blobs with no digit structure.
- **Epoch 4:** The Generator starts assembling curved loops and stroke primitives, beginning to approximate circular digits ('0', '8') and linear strokes ('1', '7').
- **Epoch 12:** Clear digit topologies (recognizable 0s, 1s, 7s, 3s) emerge with clean dark backgrounds.

### 2. The Fallacy of GAN Loss Curves:
- In supervised learning, lower loss directly implies superior generalization.
- In GANs, **losses oscillate**. As the Generator improves, the Discriminator's task becomes harder, raising its loss. When the Discriminator adjusts, the Generator's loss climbs. A steady equilibrium is ideal, and visual inspection of samples remains the ground truth of generative quality.
